# Spark Memory Architecture: Unified Memory Model

**Module**: 06_memory_and_storage  
**Notebook**: 01_spark_memory_architecture.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Overview of Spark's Unified Memory Management, JVM heap regions, and on-heap vs off-heap memory.
- **Why does this topic exist?**: Understanding memory pools is required to prevent OutOfMemoryError and eliminate disk spill.
- **What problem does it solve?**: Debugging memory allocation failures in production.
- **Where is it used in real-world Data Engineering?**: Sizing executor JVMs for enterprise Spark clusters.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | JVM basics, executors, partitioning |
| **Required depth** | Working knowledge: can explain the unified memory model and its failure modes |
| **Data Engineer** | Interpret executor memory metrics and recognize when memory pressure causes spill or OOM. |
| **Senior Data Engineer** | Size memory pools and explain why a job fails despite free-looking heap. |
| **Spark specialist** | Reason about reserved memory, user memory, unified pools, and off-heap allocation. |
| **Next topic** | `06_memory_and_storage/02_executor_memory.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spark does not give all executor heap to data processing. A reserved region, user memory, and a unified
storage/execution pool share the JVM heap, and the operating system plus JVM overhead sit outside it. Misreading
these boundaries leads to "the executor has 8 GB, why did it OOM?" incidents and to cache settings that silently
evict or spill.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Reserved memory** | Fixed safety region | Roughly 300 MB reserved for Spark internals |
| **User memory** | Non-Spark working space | Remainder after reserved and unified pools |
| **Unified pool** | Execution plus storage | Controlled by `spark.memory.fraction` |
| **Execution memory** | Sort, hash, shuffle buffers | Used by operators; can evict storage |
| **Storage memory** | Cached blocks and broadcast data | Can borrow from execution but cannot evict it |
| **Off-heap** | Memory outside the JVM heap | Optional, controlled by off-heap settings |

### How it works, step by step

1. The JVM starts with the configured executor heap.
2. Spark reserves a fixed safety region for internal objects.
3. A fraction of the remaining heap becomes the unified storage/execution pool.
4. The rest is user memory available to application objects and UDFs.
5. Execution allocations can evict cached storage blocks when the pool is under pressure.
6. Storage cannot evict active execution memory, so a cache-heavy workload can spill or fail under execution pressure.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Unified Memory Management Breakdown
```
Total Executor Memory (spark.executor.memory)
+-----------------------------------------------------------------------------------+
| Reserved Memory (300 MB hardcoded)                                                |
+-----------------------------------------------------------------------------------+
| User Memory (Default: (Heap - 300MB) * (1.0 - 0.60) = 40% of usable heap)        |
|  - User data structures, UDF allocations, internal metadata                       |
+-----------------------------------------------------------------------------------+
| Spark Memory Pool (Default: (Heap - 300MB) * 0.60 = 60% of usable heap)           |
|  +---------------------------------------+--------------------------------------+  |
|  | Storage Memory                        | Execution Memory                     |  |
|  | - Cached DataFrames/RDDs              | - Shuffles, Joins, Sorts, Aggregates |  |
|  | - Broadcast variables                 | - Spills to disk if RAM exhausted    |  |
|  +---------------------------------------+--------------------------------------+  |
+-----------------------------------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Memory_Arch_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Inspect configured memory fractions
print("spark.memory.fraction:        ", spark.conf.get("spark.memory.fraction", "0.6 (default)"))
print("spark.memory.storageFraction: ", spark.conf.get("spark.memory.storageFraction", "0.5 (default)"))
print("spark.executor.memory:        ", spark.conf.get("spark.executor.memory", "2g"))

---

## Experiment: Inspecting Memory Pool Status

- **1. What are we testing?**: Query SparkContext statusTracker for memory metrics.
- **2. What do we expect?**: Returns executor memory pools.

In [ ]:
st = spark.sparkContext.statusTracker; print('Active executor IDs:', st.getExecutorInfos())

**3. What happened?**: Queried statusTracker successfully.  
**4. Why did it happen?**: StatusTracker exposes live executor metrics.  
**5. Practical Engineering Lesson**: Inspect statusTracker to monitor live memory health.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Unexplained executor OOM

- **Industry**: Insurance
- **Scenario**: An executor with 8 GB heap fails while the Spark UI shows only a few GB of cached data.
- **Problem**: The cache is not the only consumer: user data structures, Python workers, and JVM overhead also consume memory.
- **Why this concept is relevant**: The unified model explains why heap accounting is not the same as data size; the fix may be reducing wide objects or increasing overhead.
- **Implementation idea**: Inspect GC logs, executor memory metrics, and the physical plan for large in-memory structures.
- **Expected result**: A diagnosis that distinguishes heap pressure from container-level memory limits.
- **Alternative**: Move to larger executors.
- **Trade-offs**: Larger executors reduce per-executor pressure but increase failure blast radius and GC pause cost.

### Use Case 2: Cache-heavy dashboard job

- **Industry**: Business intelligence
- **Scenario**: A job caches five intermediate DataFrames to speed up a dashboard refresh.
- **Problem**: The caches compete with the execution pool, and shuffle stages begin to spill.
- **Why this concept is relevant**: Understanding dynamic borrowing shows that caching everything can degrade the very stages it was meant to accelerate.
- **Implementation idea**: Measure cache hit ratio, evicted blocks, spill, and stage time with and without each cache.
- **Expected result**: A smaller cache set that improves end-to-end runtime.
- **Alternative**: Persist to disk or materialize a curated table.
- **Trade-offs**: Disk-backed caching reduces heap pressure but adds I/O; a curated table shifts cost to the storage layer.

---

## Business Value

- Fewer misdiagnosed OOM incidents.
- Better cache and executor sizing decisions.
- Lower infrastructure cost from oversized heaps.
- Shared vocabulary between Spark and platform engineers.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Do not equate dataset size with heap requirement; serialization and object overhead matter.
- Leave room for Python workers when using PySpark UDFs.
- Monitor GC time and spill alongside heap usage.
- Account for container limits, not just JVM heap.
- Validate memory changes with a representative workload.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Watch the unified pool under pressure

**Steps**

1. Cache a DataFrame that occupies a meaningful share of executor memory.
2. Run a shuffle-heavy aggregation that uses execution memory.
3. Observe cached blocks, spill, and GC time in the UI and logs.

**What to observe**: Execution allocations can evict cached blocks; storage borrowing reverses when execution needs memory.

**Healthy result**: The job completes with bounded spill and a measurable cache benefit.

**Problematic result**: The cache is almost entirely evicted and the job slows down.

**Engineering takeaway**: Caching is a budget decision, not a free speedup.

---

## Trade-offs and Alternatives

### Advantages

- Unified pool adapts to workload
- Caching and execution share capacity
- Well instrumented

### Limitations

- Complex to reason about
- Python and off-heap memory sit outside the model
- OOM can occur below the heap limit

### When to use it

- Executor sizing
- Cache planning
- Spill diagnosis

### When not to use it

- Assuming free heap means safe
- Caching every intermediate
- Ignoring container limits

### Alternatives inside Spark

- Unified memory manager
- Storage levels
- Off-heap memory

### Alternatives outside Spark

- Container memory limits
- JVM GC tuning
- Kubernetes pod sizing

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Calculate Reserved Memory
**Problem Statement**: What is the reserved memory in Spark?  
**Input / Context**: Concept

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Calculate Usable Pool
**Problem Statement**: Calculate Spark pool for 4GB heap.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Off-heap Config
**Problem Statement**: Check off-heap config.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The 8 GB executor that needs 12 GB

**Scenario**: An executor configured with 8 GB heap is killed by the container for exceeding its memory limit.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check `spark.executor.memoryOverhead` and the container limit.
2. Look for Python worker memory or off-heap allocations.
3. Compare the pod/container limit with heap plus overhead plus OS reserve.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Calculate Reserved Memory):
```python
print('300 MB')
```

#### Solution 2 (Calculate Usable Pool):
```python
print(f'Spark pool: {(4096 - 300) * 0.6:.1f} MB')
```

#### Solution 3 (Off-heap Config):
```python
print(spark.conf.get('spark.memory.offHeap.enabled','false'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Unified Memory**: Storage and execution dynamically borrow from a shared pool.

### Essential Technical Terms
- `Unified Memory`: Dynamically shared memory pool.
- `Off-Heap`: Memory outside JVM garbage collector.

### Common Engineering Mistakes
- **Assuming all heap is available for caching**: Only a fraction of the heap is initially reserved for storage (roughly 30% with defaults); storage can borrow unused execution memory, but it is not a fixed guarantee.

### Interview Questions on This Topic
- **Q: Can execution memory evict storage memory?**
  - *A*: Yes, execution memory can evict storage blocks down to storageFraction.


### More Interview Questions

- **Q: What is the unified memory model?**
  - *A*: A single pool shared by execution and storage memory, where execution can evict storage and storage can borrow unused execution capacity, subject to configured fractions.
- **Q: Why can an executor OOM with free heap?**
  - *A*: Memory outside the Java heap, such as Python workers, off-heap buffers, JVM overhead, and the container limit, can be exhausted even when heap usage looks moderate.

### What to Study Next
Open 02_executor_memory.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')